# AIMO3 Baseline Notebook — Day3 Minimal Real Solver


## Cell 1: 設定・定数定義（Day3: real solver 1問実行）


In [ ]:
# ============================================================
# 設定・定数
# ============================================================
import os

# Day3 は real solver 1問実行の確認日
SOLVER_TYPE = "real"  # "placeholder" | "real"
assert SOLVER_TYPE == "real", (
    f"Day3 は real solver のみ許可。現在: {SOLVER_TYPE!r}"
)

# Kaggle データセットパス候補（存在確認は Cell 3 で実施）
DATASET_PATH_CANDIDATES = [
    "/kaggle/input/ai-mathematical-olympiad-progress-prize-3",
    "/kaggle/input/aimo-2025-ai-mathematical-olympiad-progress",
    "/kaggle/input/aimo3",
]

# モデルパス候補（存在確認は Cell 4 で実施）
MODEL_PATH_CANDIDATES = [
    "/kaggle/input/nvidia-openmath-nemotron-14b",
    "/kaggle/input/openmath-nemotron-14b-kaggle",
    "/kaggle/input/numina-math-7b-tir",
    "/kaggle/input/deepseek-math-7b-instruct",
    "/kaggle/input/qwen2-5-math-7b-instruct",
    "/kaggle/input/internlm2-math-plus-7b",
]

# 出力パス
OUTPUT_DIR = "/kaggle/working"
SUBMISSION_PATH = os.path.join(OUTPUT_DIR, "submission.csv")
REAL_ONE_PATH = os.path.join(OUTPUT_DIR, "real_solver_one_problem.json")
REAL_ERROR_PATH = os.path.join(OUTPUT_DIR, "real_solver_error.json")

print(f"SOLVER_TYPE     : {SOLVER_TYPE}")
print(f"OUTPUT_DIR      : {OUTPUT_DIR}")
print(f"SUBMISSION_PATH : {SUBMISSION_PATH}")
print(f"REAL_ONE_PATH   : {REAL_ONE_PATH}")
print(f"REAL_ERROR_PATH : {REAL_ERROR_PATH}")


## Cell 2: 環境確認（Python / GPU / CUDA / torch）

In [ ]:
# ============================================================
# 環境確認
# BLOCKED 条件: import 失敗が 1件でもあれば BLOCKED
# ============================================================
import sys
import subprocess

print("=" * 50)
print("Python:", sys.version)
print("Platform:", sys.platform)

# pandas
try:
    import pandas as pd
    print(f"pandas    : {pd.__version__}  OK")
except ImportError as e:
    print(f"pandas    : MISSING — {e}")
    print("BLOCKED: pandas not available")

# torch
try:
    import torch
    print(f"torch     : {torch.__version__}  OK")
    cuda_available = torch.cuda.is_available()
    print(f"CUDA avail: {cuda_available}")
    if cuda_available:
        print(f"CUDA ver  : {torch.version.cuda}")
        print(f"GPU count : {torch.cuda.device_count()}")
        for i in range(torch.cuda.device_count()):
            name = torch.cuda.get_device_name(i)
            mem  = torch.cuda.get_device_properties(i).total_memory / 1e9
            print(f"  GPU[{i}]   : {name}  ({mem:.1f} GB)")
    else:
        print("WARNING: CUDA not available — GPU inference impossible")
except ImportError as e:
    print(f"torch     : MISSING — {e}")
    print("BLOCKED: torch not available")

# transformers (モデル推論に必要。Day2 では import 確認のみ)
try:
    import transformers
    print(f"transformers: {transformers.__version__}  OK")
except ImportError as e:
    print(f"transformers: MISSING — {e}")
    print("WARNING: transformers not available (needed for Day3+ real solver)")

# nvidia-smi
try:
    smi = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.free",
         "--format=csv,noheader"],
        capture_output=True, text=True, timeout=10
    )
    if smi.returncode == 0:
        print("nvidia-smi:", smi.stdout.strip())
    else:
        print("nvidia-smi: not available")
except Exception as e:
    print(f"nvidia-smi: error — {e}")

print("=" * 50)

## Cell 3: Dataset path 確認

In [ ]:
# ============================================================
# Dataset path 確認
# BLOCKED 条件: DATASET_PATH が None のまま → BLOCKED
# ============================================================
import os
import glob

DATASET_PATH = None

print("=== Dataset path 候補 ====")
for candidate in DATASET_PATH_CANDIDATES:
    exists = os.path.isdir(candidate)
    print(f"  {'OK' if exists else '--'}  {candidate}")
    if exists and DATASET_PATH is None:
        DATASET_PATH = candidate

print()
if DATASET_PATH is None:
    print("BLOCKED: dataset path が見つかりません。")
    print("  → Kaggle で Add Data から公式 dataset を追加してください。")
    raise RuntimeError("BLOCKED: dataset path not found")
else:
    print(f"DATASET_PATH: {DATASET_PATH}")
    all_files = glob.glob(os.path.join(DATASET_PATH, "**", "*"), recursive=True)
    print("ファイル一覧:")
    for f in sorted(all_files):
        size = os.path.getsize(f) if os.path.isfile(f) else "-"
        print(f"  {f}  ({size} bytes)")

# 必須ファイルの確認
for required in ["test.csv", "sample_submission.csv"]:
    p = os.path.join(DATASET_PATH, required)
    if os.path.isfile(p):
        print(f"  FOUND: {required}")
    else:
        print(f"  MISSING: {required} — BLOCKED")
        raise FileNotFoundError(f"BLOCKED: {required} not found in {DATASET_PATH}")

## Cell 4: Model path 確認（Day3: 必須）


In [ ]:
# ============================================================
# Model path 確認
# Day3 では real solver 用 model path を必須にする
# ============================================================

MODEL_PATH = None

print("=== Model path 候補 ===")
for candidate in MODEL_PATH_CANDIDATES:
    exists = os.path.isdir(candidate)
    print(f"  {'OK' if exists else '--'}  {candidate}")
    if exists and MODEL_PATH is None:
        MODEL_PATH = candidate
        files = os.listdir(candidate)[:5]
        print(f"       → 先頭5ファイル: {files}")

print()
if MODEL_PATH is None:
    raise RuntimeError(
        "BLOCKED: model path が見つかりません。Day3 real solver ではモデルが必須です。"
    )

print(f"MODEL_PATH: {MODEL_PATH}")
print("Day3 model path check PASS")


## Cell 4.5: real solver model load（最小）


In [ ]:
# ============================================================
# real solver 用 model load
# silent fallback 禁止: load 失敗時は例外で停止
# ============================================================
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print("=== real model load ===")
print(f"model_path: {MODEL_PATH}")

dtype = torch.float16 if torch.cuda.is_available() else torch.float32
print(f"torch_dtype: {dtype}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=dtype,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
print("real model load PASS")


## Cell 5: Output path 確認

In [ ]:
# ============================================================
# Output path 確認
# BLOCKED 条件: /kaggle/working が書き込み不可 → BLOCKED
# ============================================================

print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"  exists   : {os.path.isdir(OUTPUT_DIR)}")
print(f"  writable : {os.access(OUTPUT_DIR, os.W_OK)}")

if not os.path.isdir(OUTPUT_DIR):
    print("BLOCKED: OUTPUT_DIR が存在しません")
    raise RuntimeError(f"BLOCKED: {OUTPUT_DIR} does not exist")

if not os.access(OUTPUT_DIR, os.W_OK):
    print("BLOCKED: OUTPUT_DIR が書き込み不可です")
    raise RuntimeError(f"BLOCKED: {OUTPUT_DIR} is not writable")

# ログ保存先の確認
LOG_DIR = os.path.join(OUTPUT_DIR, "logs")
os.makedirs(LOG_DIR, exist_ok=True)
print(f"LOG_DIR   : {LOG_DIR}  (作成済み)")
print("Output path PASS")

## Cell 6: AIMO3 データ読み込み

In [ ]:
# ============================================================
# AIMO3 公式データ読み込み
# BLOCKED 条件: DATASET_PATH が None → Cell 3 で止まっているはず
# ============================================================
import pandas as pd

test_path   = os.path.join(DATASET_PATH, "test.csv")
sample_path = os.path.join(DATASET_PATH, "sample_submission.csv")

test_df   = pd.read_csv(test_path)
sample_df = pd.read_csv(sample_path)

print("=== test.csv ===")
print(f"  shape  : {test_df.shape}")
print(f"  columns: {list(test_df.columns)}")
print(f"  dtypes : {test_df.dtypes.to_dict()}")
print()
print("先頭 3 行:")
print(test_df.head(3).to_string())

print()
print("=== sample_submission.csv ===")
print(f"  shape  : {sample_df.shape}")
print(f"  columns: {list(sample_df.columns)}")
print()
print(sample_df.head(3).to_string())

# 問題数・ID 確認
NUM_PROBLEMS = len(test_df)
print(f"\n問題数 (NUM_PROBLEMS): {NUM_PROBLEMS}")

# 問題テキストのカラム名を自動検出
PROBLEM_COL = None
ANSWER_COL  = "answer"
ID_COL      = "id"

for candidate in ["problem", "problem_text", "question", "text"]:
    if candidate in test_df.columns:
        PROBLEM_COL = candidate
        break

if PROBLEM_COL is None:
    print(f"BLOCKED: problem カラムが見つかりません。columns={list(test_df.columns)}")
    raise RuntimeError("BLOCKED: problem column not found")

if ID_COL not in test_df.columns:
    print(f"BLOCKED: id カラムが見つかりません。columns={list(test_df.columns)}")
    raise RuntimeError("BLOCKED: id column not found")

print(f"ID_COL      : {ID_COL!r}")
print(f"PROBLEM_COL : {PROBLEM_COL!r}")

## Cell 7: 1問取り出し・問題文確認

In [ ]:
# ============================================================
# 1問取り出し・問題文確認
# ============================================================

FIRST_ROW   = test_df.iloc[0]
FIRST_ID    = FIRST_ROW[ID_COL]
FIRST_PROB  = FIRST_ROW[PROBLEM_COL]

print("=== 問題 #0 ===")
print(f"  id     : {FIRST_ID!r}")
print(f"  problem: {FIRST_PROB[:300]}{'...' if len(str(FIRST_PROB)) > 300 else ''}")
print(f"  (全長 {len(str(FIRST_PROB))} 文字)")

## Cell 8: Real solver 定義（1問専用・最小）


In [ ]:
# ============================================================
# Real solver
# Day3 最小版: 1問のみ対象。silent fallback は行わない。
# ============================================================
import re


def solve(problem_id: str, problem_text: str) -> int:
    assert SOLVER_TYPE == "real", (
        f"solver_type mismatch: expected 'real', got {SOLVER_TYPE!r}"
    )

    prompt = (
        "Solve the following AIME problem. "
        "Return only the final integer answer between 0 and 999.\n\n"
        f"Problem ID: {problem_id}\n"
        f"Problem:\n{problem_text}\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=96,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    tail = text[len(prompt):].strip() if text.startswith(prompt) else text

    m = re.search(r"(?<!\d)(\d{1,3})(?!\d)", tail)
    if m is None:
        raise ValueError(f"real solver parse failed. raw_output={tail[:400]!r}")

    answer = int(m.group(1))
    if not (0 <= answer <= 999):
        raise ValueError(f"AIME answer out of range: {answer}")
    return answer

print("Real solver definition PASS")


## Cell 9: real solver 1問実行 + runtime + 例外保存


In [ ]:
# ============================================================
# real solver 1問実行 + runtime
# 失敗時は例外内容を JSON 保存して再送出
# ============================================================
import json
import time
import traceback

try:
    start = time.monotonic()
    answer_0 = solve(FIRST_ID, FIRST_PROB)
    elapsed = time.monotonic() - start

    one_result = {
        "id": FIRST_ID,
        "answer": int(answer_0),
        "elapsed_sec": round(elapsed, 6),
        "solver_type": SOLVER_TYPE,
        "model_path": MODEL_PATH,
    }

    with open(REAL_ONE_PATH, "w", encoding="utf-8") as f:
        json.dump(one_result, f, ensure_ascii=False, indent=2)

    print("=== real solver 1問実行 ===")
    print(one_result)
    print(f"saved: {REAL_ONE_PATH}")
except Exception as e:
    err = {
        "id": str(FIRST_ID),
        "solver_type": SOLVER_TYPE,
        "model_path": MODEL_PATH,
        "error_type": type(e).__name__,
        "error_message": str(e),
        "traceback": traceback.format_exc(),
    }
    with open(REAL_ERROR_PATH, "w", encoding="utf-8") as f:
        json.dump(err, f, ensure_ascii=False, indent=2)
    print(f"saved error: {REAL_ERROR_PATH}")
    raise


## Cell 10: Day3 保存物確認（1問のみ）


In [ ]:
# ============================================================
# Day3 保存物確認（1問のみ）
# ============================================================
print("=== Day3 artifacts ===")
print(f"real result exists : {os.path.isfile(REAL_ONE_PATH)}")
print(f"error log exists   : {os.path.isfile(REAL_ERROR_PATH)}")
if os.path.isfile(REAL_ONE_PATH):
    print(f"real result size   : {os.path.getsize(REAL_ONE_PATH)} bytes")
if os.path.isfile(REAL_ERROR_PATH):
    print(f"error log size     : {os.path.getsize(REAL_ERROR_PATH)} bytes")


## Cell 11: Day3 実行結果の再読込確認（1問）


In [ ]:
# ============================================================
# Day3 実行結果の再読込確認（1問）
# ============================================================
import json

if not os.path.isfile(REAL_ONE_PATH):
    raise RuntimeError(f"BLOCKED: {REAL_ONE_PATH} が存在しません")

with open(REAL_ONE_PATH, "r", encoding="utf-8") as f:
    saved = json.load(f)

print("=== real_solver_one_problem.json ===")
print(saved)
assert saved.get("id") == FIRST_ID
assert isinstance(saved.get("answer"), int)
assert 0 <= saved["answer"] <= 999
print("Day3 one-problem result validation PASS")


## Cell 12: Day3 完了チェック（real solver 1問）


In [ ]:
# ============================================================
# Day3 完了チェック
# このセルが正常終了 = Day3 PASS（real solver 1問）
# ============================================================
import json

checks = {
    "solver_type": SOLVER_TYPE,
    "dataset_path": DATASET_PATH,
    "model_path": MODEL_PATH,
    "real_result_exists": os.path.isfile(REAL_ONE_PATH),
    "real_error_exists": os.path.isfile(REAL_ERROR_PATH),
}

print("=== Day3 完了チェック ===")
for key, val in checks.items():
    print(f"  {key}: {val}")

if SOLVER_TYPE != "real":
    raise RuntimeError("Day3 BLOCKED: solver_type must be 'real'")
if not os.path.isfile(REAL_ONE_PATH):
    raise RuntimeError("Day3 BLOCKED: one-problem real solver output missing")

with open(REAL_ONE_PATH, "r", encoding="utf-8") as f:
    row = json.load(f)

if row.get("solver_type") != "real":
    raise RuntimeError(f"Day3 BLOCKED: solver_type mismatch in artifact: {row.get('solver_type')!r}")

print("Day3: PASS — real solver 1問実行成立")


---

## 注意事項

| 項目 | 説明 |
|---|---|
| `solver_type` | `"placeholder"` のみ許可（Day2）。`"real"` への変更は Day3 以降 |
| `submission.csv` | `answer` は常に `0`（placeholder）— public score は 0/50 |
| AIME answer 範囲 | 整数 0〜999。Kaggle では plain int で submit |
| runtime 確認 | Cell 9 の換算値でタイムバジェットを確認すること |
| model | Day3 以降: `nvidia/OpenMath-Nemotron-14B-Kaggle` (AIMO2 優勝) を推奨 |

## Day3 以降の手順

1. Cell 1 の `SOLVER_TYPE` を `"real"` に変更
2. Cell 4 でモデルパスを確認
3. Cell 8 の `solve()` を実 solver に差し替え
4. Cell 9 でタイマー計測（32問換算 < runtime 制限を確認）
5. 全件実行 → submission.csv 生成 → Kaggle submit